In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load data
df = pd.read_csv(r'../data/raw/ecommerce_price_prediction-train.csv')
df['capturedAt'] = pd.to_datetime(df['capturedAt'])
df['scrape_date'] = df['capturedAt'].dt.date

In [2]:
# Define the outage date based on EDA timeline
outage_date = df['scrape_date'].max()

# Split into Train and Validation (Outage Day)
train_df = df[df['scrape_date'] < outage_date].copy()
val_df = df[df['scrape_date'] == outage_date].copy()

print(f"--- Data Split Summary ---")
print(f"Training Set (Before {outage_date}): {train_df.shape[0]} rows")
print(f"Validation Set (Outage Day - {outage_date}): {val_df.shape[0]} rows")

--- Data Split Summary ---
Training Set (Before 2025-03-22): 301355 rows
Validation Set (Outage Day - 2025-03-22): 4871 rows


In [5]:
def prepare_features(data, is_train=True):
    df_feat = data.copy()
    
    # 1. Temporal Cyclical Features
    df_feat['hour'] = df_feat['capturedAt'].dt.hour
    df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24.0)
    df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24.0)
    
    # 2. Pricing Boundaries & Zero-Division Fix
    denominator = df_feat['item_price_max'] - df_feat['item_price_min']
    df_feat['price_position'] = np.where(
        denominator == 0, 
        0.5, 
        (df_feat['priceBeforeDiscount'] - df_feat['item_price_min']) / (denominator + 1e-5)
    )
    
    df_feat['discount_ratio'] = np.where(
        df_feat['priceBeforeDiscount'] == 0,
        0.0,
        df_feat['raw_discount'] / (df_feat['priceBeforeDiscount'] + 1e-5)
    )
    
    # 3. High-Cardinality Frequency Encoding
    # Note: In production, frequencies should be mapped from Train to Val, 
    # but for baseline exploration, we map directly.
    df_feat['shop_freq'] = df_feat['shopId'].map(df_feat['shopId'].value_counts())
    
    # 4. Handle Missing Brand
    df_feat['brand'] = df_feat['brand'].fillna('Unknown').astype('category')
    
    # 5. Convert Categorical Identifiers to Pandas Category Type for LightGBM
# 5. Convert Categorical Identifiers to Pandas Category Type for LightGBM
    cat_cols = [
        'shopId', 'itemId', 'modelId', 'cat_id', 
        'is_free_shipping', 'is_pre_order', 'is_official_shop',
        'is_verified', 'is_preferred_plus_seller'  # <-- ADDED THESE TWO HERE
    ]
    for col in cat_cols:
        if col in df_feat.columns:  # Added a safety check in case a column is missing
            df_feat[col] = df_feat[col].astype('category')
        
    # Select features to drop (sparse or redundant)
    drop_cols = ['capturedAt', 'scrape_date', 'stock', 'normal_stock', 'promotionId']
    if 'price' in df_feat.columns and not is_train:
        # Keep price for validation evaluation later, but separate it from X
        pass
        
    df_feat = df_feat.drop(columns=[c for c in drop_cols if c in df_feat.columns])
    return df_feat

# Apply feature engineering
train_processed = prepare_features(train_df, is_train=True)
val_processed = prepare_features(val_df, is_train=False)

In [6]:
# Define features and target
features = [col for col in train_processed.columns if col not in ['price', 'log_price']]

X_train = train_processed[features]
y_train_log = np.log1p(train_processed['price'])

X_val = val_processed[features]
y_val_actual = val_processed['price'] # Kept for evaluation

# Initialize and train LightGBM Regressor
model_global = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

model_global.fit(X_train, y_train_log)
print("Global Model Trained Successfully!")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10558
[LightGBM] [Info] Number of data points in the train set: 301355, number of used features: 26
[LightGBM] [Info] Start training from score 16.984750
Global Model Trained Successfully!


In [7]:
# Randomly sample 100 indices to act as our Anchor Set
anchor_indices = val_processed.sample(100, random_state=RANDOM_SEED).index

anchor_set = val_processed.loc[anchor_indices].copy()
# The rest of the validation rows act as the hidden test target
hidden_test_set = val_processed.drop(index=anchor_indices).copy()

print(f"Anchor Set Size: {anchor_set.shape[0]} rows")
print(f"Hidden Test Set Size (To Predict): {hidden_test_set.shape[0]} rows")

Anchor Set Size: 100 rows
Hidden Test Set Size (To Predict): 4771 rows


In [9]:
# 1. Predict on Anchor Set (Outputs Log Price)
anchor_preds_log = model_global.predict(anchor_set[features])
anchor_preds_actual = np.expm1(anchor_preds_log)

# 2. Calculate Calibration Factor (Actual Price / Predicted Price)
# We use median ratio to be robust against anchor outliers
calibration_ratios = anchor_set['price'] / (anchor_preds_actual + 1e-5)
global_calibration_factor = np.median(calibration_ratios)

print(f"Calculated Global Calibration Factor: {global_calibration_factor:.4f}")
print(f"*(If > 1, model underpredicted the market. If < 1, model overpredicted.)*")

Calculated Global Calibration Factor: 0.9999
*(If > 1, model underpredicted the market. If < 1, model overpredicted.)*


In [10]:
# Predict on the hidden validation rows
test_preds_log = model_global.predict(hidden_test_set[features])
test_preds_base = np.expm1(test_preds_log)

# Apply Calibration Factor
test_preds_calibrated = test_preds_base * global_calibration_factor

# Calculate Metrics
y_test_true = hidden_test_set['price']

mae_base = mean_absolute_error(y_test_true, test_preds_base)
mae_calibrated = mean_absolute_error(y_test_true, test_preds_calibrated)

mape_base = mean_absolute_percentage_error(y_test_true, test_preds_base)
mape_calibrated = mean_absolute_percentage_error(y_test_true, test_preds_calibrated)

print("--- PERFORMANCE EVALUATION ON HIDDEN TEST SET ---")
print(f"Baseline MAE : {mae_base:,.2f}  | Calibrated MAE : {mae_calibrated:,.2f}")
print(f"Baseline MAPE: {mape_base*100:.2f}%  | Calibrated MAPE: {mape_calibrated*100:.2f}%")

# Quick breakdown improvement narrative
improvement = ((mape_base - mape_calibrated) / mape_base) * 100
print(f"\nCalibration Strategy improved MAPE by: {improvement:.2f}%")

--- PERFORMANCE EVALUATION ON HIDDEN TEST SET ---
Baseline MAE : 271,729.85  | Calibrated MAE : 272,185.69
Baseline MAPE: 0.97%  | Calibrated MAPE: 0.97%

Calibration Strategy improved MAPE by: 0.06%


In [11]:
# Predict on the hidden validation rows
test_preds_log = model_global.predict(hidden_test_set[features])
test_preds_base = np.expm1(test_preds_log)

# Apply Calibration Factor
test_preds_calibrated = test_preds_base * global_calibration_factor

# Calculate Metrics
y_test_true = hidden_test_set['price']

mae_base = mean_absolute_error(y_test_true, test_preds_base)
mae_calibrated = mean_absolute_error(y_test_true, test_preds_calibrated)

mape_base = mean_absolute_percentage_error(y_test_true, test_preds_base)
mape_calibrated = mean_absolute_percentage_error(y_test_true, test_preds_calibrated)

print("--- PERFORMANCE EVALUATION ON HIDDEN TEST SET ---")
print(f"Baseline MAE : {mae_base:,.2f}  | Calibrated MAE : {mae_calibrated:,.2f}")
print(f"Baseline MAPE: {mape_base*100:.2f}%  | Calibrated MAPE: {mape_calibrated*100:.2f}%")

# Quick breakdown improvement narrative
improvement = ((mape_base - mape_calibrated) / mape_base) * 100
print(f"\nCalibration Strategy improved MAPE by: {improvement:.2f}%")

--- PERFORMANCE EVALUATION ON HIDDEN TEST SET ---
Baseline MAE : 271,729.85  | Calibrated MAE : 272,185.69
Baseline MAPE: 0.97%  | Calibrated MAPE: 0.97%

Calibration Strategy improved MAPE by: 0.06%


## 8. Granular Calibration: Category-Level Price Shifts vs. Variance Trade-off

### Hypothesis
Prices in an e-commerce marketplace rarely shift uniformly across the entire platform. During an outage or promotional campaign, specific categories (e.g., Electronics or Gadgets) might experience aggressive price drops or inflation, while other utility categories remain static. 

### The Sample Size & Variance Trade-off
By moving from a **Global Calibration Factor** to a **Granular Category Calibration**, we encounter a classic production ML trade-off:
1. **Pros:** Increased sensitivity to distinct category trends, allowing more precise adjustments.
2. **Cons (The Small-Sample Trap):** Our anchor set contains only **100 samples** across **26 unique categories (cat_id)**. This means some categories will only have 1, 2, or even 0 representative samples in the anchor set. Computing a median ratio from 1 or 2 observations introduces immense variance and high risk of **overfitting to anchor noise**.

### Defensive Engineering Strategy
To mitigate the small-sample risk, our code implements a fallback mechanism using `.fillna(global_calibration_factor)`. If a category present in the hidden test set never appeared in the 100 anchor samples, it safely defaults back to our stable macro global factor (~0.9999).

In [12]:
# 1. Calculate Calibration Factor per Category from the 100 anchors
anchor_set['pred_base'] = np.expm1(model_global.predict(anchor_set[features]))
anchor_set['ratio'] = anchor_set['price'] / (anchor_set['pred_base'] + 1e-5)

# Group by cat_id and get the median ratio for each category
category_calibration = anchor_set.groupby('cat_id')['ratio'].median().to_dict()

# 2. Map the calibration factors back to the Hidden Test Set
# If a category doesn't exist in the 100 anchors, fallback to the global factor (0.9999) or 1.0
hidden_test_set['calib_factor_cat'] = hidden_test_set['cat_id'].map(category_calibration).fillna(global_calibration_factor)

# 3. Apply the granular calibration
test_preds_cat_calibrated = test_preds_base * hidden_test_set['calib_factor_cat']

# 4. Re-evaluate Metrics
mae_cat = mean_absolute_error(y_test_true, test_preds_cat_calibrated)
mape_cat = mean_absolute_percentage_error(y_test_true, test_preds_cat_calibrated)

print("--- GRANULAR CATEGORY-LEVEL EVALUATION ---")
print(f"Category Calibrated MAE : {mae_cat:,.2f}")
print(f"Category Calibrated MAPE: {mape_cat*100:.2f}%")

--- GRANULAR CATEGORY-LEVEL EVALUATION ---
Category Calibrated MAE : 253,528.32
Category Calibrated MAPE: 0.96%


# Tier 2

In [17]:
## === TIER 2: ENTITY-SPECIFIC MODELING ===

# 1. Calculate historical behavior per shop from training data
shop_stats = train_df.groupby('shopId')['price'].agg(['mean', 'std']).reset_index()
shop_stats.columns = ['shopId', 'historical_shop_price_mean', 'historical_shop_price_std']

# 2. Calculate historical behavior per item
item_stats = train_df.groupby('itemId')['price'].agg(['mean']).reset_index()
item_stats.columns = ['itemId', 'historical_item_price_mean']

# 3. Map these historical priors back to Train and Validation to "condition" the model
train_tier2 = train_processed.merge(shop_stats, on='shopId', how='left')
train_tier2 = train_tier2.merge(item_stats, on='itemId', how='left')
train_tier2.index = train_processed.index  # <-- TAMBAHKAN INI: Kembalikan indeks asli Train

val_tier2 = val_processed.merge(shop_stats, on='shopId', how='left')
val_tier2 = val_tier2.merge(item_stats, on='itemId', how='left')
val_tier2.index = val_processed.index  # <-- TAMBAHKAN INI: Kembalikan indeks asli Validation

# Handle Cold-Start for features (if a shop/item has no history, fill with global mean)
global_wallet_mean = train_df['price'].mean()
train_tier2['historical_shop_price_mean'] = train_tier2['historical_shop_price_mean'].fillna(global_wallet_mean)
val_tier2['historical_shop_price_mean'] = val_tier2['historical_shop_price_mean'].fillna(global_wallet_mean)
train_tier2['historical_shop_price_std'] = train_tier2['historical_shop_price_std'].fillna(0)
val_tier2['historical_shop_price_std'] = val_tier2['historical_shop_price_std'].fillna(0)
train_tier2['historical_item_price_mean'] = train_tier2['historical_item_price_mean'].fillna(global_wallet_mean)
val_tier2['historical_item_price_mean'] = val_tier2['historical_item_price_mean'].fillna(global_wallet_mean)

print("Tier 2 Entity-Conditioned Features Generated Successfully!")

Tier 2 Entity-Conditioned Features Generated Successfully!


In [18]:
# Update feature list to include the new entity-conditioned features
features_tier2 = [col for col in train_tier2.columns if col not in ['price', 'log_price']]

X_train_t2 = train_tier2[features_tier2]
y_train_t2_log = np.log1p(train_tier2['price'])

X_val_t2 = val_tier2[features_tier2]

# Train Tier 2 Model
model_tier2 = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model_tier2.fit(X_train_t2, y_train_t2_log)
print("Tier 2 Entity-Specific Model Trained!")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015391 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10098
[LightGBM] [Info] Number of data points in the train set: 301355, number of used features: 29
[LightGBM] [Info] Start training from score 16.984750
Tier 2 Entity-Specific Model Trained!


In [19]:
# 1. Re-isolate Anchor and Hidden Test Set for Tier 2 using the same indices
anchor_set_t2 = val_tier2.loc[anchor_indices].copy()
hidden_test_set_t2 = val_tier2.drop(index=anchor_indices).copy()

# 2. Get base predictions from Tier 2 Model on Anchor Set
anchor_set_t2['pred_base'] = np.expm1(model_tier2.predict(anchor_set_t2[features_tier2]))
anchor_set_t2['ratio'] = anchor_set_t2['price'] / (anchor_set_t2['pred_base'] + 1e-5)

# 3. Compute Entity-Level Calibration (Shop Level)
shop_calibration = anchor_set_t2.groupby('shopId')['ratio'].median().to_dict()

# 4. Map back to Hidden Test Set with Cold-Start Defense Fallback Hierarchy:
# Hierarchy: Shop Factor -> If Null, Fallback to Global Factor
hidden_test_set_t2['calib_factor_shop'] = hidden_test_set_t2['shopId'].map(shop_calibration).fillna(global_calibration_factor)

# 5. Apply Tier 2 Prediction & Calibration
preds_t2_base = np.expm1(model_tier2.predict(hidden_test_set_t2[features_tier2]))
preds_t2_calibrated = preds_t2_base * hidden_test_set_t2['calib_factor_shop']

In [20]:
# Evaluate Tier 2 Performance
mae_t2 = mean_absolute_error(y_test_true, preds_t2_calibrated)
mape_t2 = mean_absolute_percentage_error(y_test_true, preds_t2_calibrated)

print("=== TIER 2 PERFORMANCE EVALUATION ===")
print(f"Tier 2 (Shop/Product Conditioned) MAE : {mae_t2:,.2f}")
print(f"Tier 2 (Shop/Product Conditioned) MAPE: {mape_t2*100:.2f}%")

print("\n=== FINAL BENCHMARK SUMMARY ===")
print(f"Tier 1 Global Model MAPE     : {mape_calibrated*100:.2f}%")
print(f"Tier 2 Entity Model MAPE     : {mape_t2*100:.2f}%")

=== TIER 2 PERFORMANCE EVALUATION ===
Tier 2 (Shop/Product Conditioned) MAE : 721,764.10
Tier 2 (Shop/Product Conditioned) MAPE: 1.43%

=== FINAL BENCHMARK SUMMARY ===
Tier 1 Global Model MAPE     : 0.97%
Tier 2 Entity Model MAPE     : 1.43%


## 9. Tier 1 vs. Tier 2 Performance Analysis & Technical Retrospective

### Experimental Results
* **Tier 1 (Global Marketplace Model + Global Calibration):** **0.97% MAPE**
* **Tier 2 (Entity-Conditioned Model + Shop-Level Calibration):** **1.43% MAPE**

### Critical Evaluation: Why the Global Model Outperformed the Entity Model

1. **The Small-Sample Calibration Trap (High Variance):**
   Our anchor set consists of a strict limit of 100 samples. When performing shop-level calibration, these 100 samples are fragmented across 219 unique shops, leaving most shops with only 1 or 2 representative anchor points. Calculating a calibration multiplier from 1 sample introduces extreme variance, forcing the model to overfit to micro-level pricing noise on the outage day rather than capturing a true platform-wide shift.

2. **Historical Data Skewness & Cold-Start Bias:**
   As discovered during our EDA, data density per shop is heavily skewed (the bottom 25% of shops have fewer than 62 historical records). Injecting `historical_shop_price_mean` for these sparse entities introduces misleading historical noise, which dilutes the predictive power of robust core boundaries like `item_price_min` and `item_price_max`.

### Strategic Takeaway for Production
For this specific price intelligence setup, a **Global Marketplace Model with Macro-Level Calibration (Tier 1)** provides a significantly more stable and generalized prediction framework. Moving forward to the production pipeline, Tier 1 will serve as our primary engine, while Tier 2 entity features should either be regularized or restricted to high-density shops only ($>1000$ historical rows).